In [ ]:
#!pip install xgboost scikit-learn pandas numpy joblib

import pandas as pd
shelter_df = pd.read_csv("TorontoShelterUsage2020_Final.csv")
shelter_df.head()

,POPULATION,AREA,OCCUPANCY,CAPACITY,Min_Temp_C,Max_Temp_C,Unemployment_Rate
0,7790,1.4,1341,1466,-1.2,1.0,5.0
1,34635,3.7,124,124,-1.2,1.0,5.0
2,29180,2.8,69,76,-1.2,1.0,5.0
3,27870,7.4,318,362,-1.2,1.0,5.0
4,34650,14.9,23,28,-1.2,1.0,5.0


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
import joblib

# Load cleaned data
model_df_tuned = pd.read_csv("TorontoShelterUsage2020_Final.csv")

# Replace infinities with NaN, drop rows where target is missing
model_df_tuned.replace([np.inf, -np.inf], np.nan, inplace=True)
model_df_tuned.dropna(subset=["OCCUPANCY_Rate"], inplace=True)

# Feature engineering
if "Max_Temp_C" in model_df_tuned.columns and "Min_Temp_C" in model_df_tuned.columns:
    model_df_tuned["Temp_Range"] = model_df_tuned["Max_Temp_C"] - model_df_tuned["Min_Temp_C"]

# Split predictors from target
X_tuned = model_df_tuned.drop(columns=["OCCUPANCY_Rate"])
y_tuned = model_df_tuned["OCCUPANCY_Rate"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tuned, y_tuned, test_size = 0.2, random_state = 42
)

# Detect column types
categorical_cols = X_tuned.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X_tuned.select_dtypes(include=["number"]).columns.tolist()

# Build preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10)),
])
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])
preprocessor = ColumnTransformer(transformers=[
    ("cat", categorical_transformer, categorical_cols),
    ("num", numeric_transformer, numeric_cols),
])

# Build full pipeline
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(random_state=42, n_jobs=-1)),
])

# Hyperparameter search space
param_dist = {
    "regressor__n_estimators": [100,200,300,500],
    "regressor__max_depth": [3,4,5,6],
    "regressor__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "regressor__subsample": [0.7, 0.8, 0.9, 1.0],
    "regressor__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "regressor__reg_alpha": [0, 0.1, 0.5],
    "regressor__reg_lambda": [1, 1.5, 2],
}

# Run randomized search
search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=30,
    cv=4,
    scoring="r2",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

print("Fitting model - this will take a few minutes...")
search.fit(X_train, y_train)

# Evaluate
y_pred = search.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n── Results ───────────────────────────────────")
print(f"Best params: {search.best_params_}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

# Save model
joblib.dump(search.best_estimator_, "best_xgboost_occupancy_rate_model_tuned.joblib")
print("\nModel saved!")

Fitting model - this will take a few minutes...
Fitting 4 folds for each of 30 candidates, totalling 120 fits

── Results ───────────────────────────────────
Best params: {'regressor__subsample': 1.0, 'regressor__reg_lambda': 2, 'regressor__reg_alpha': 0.1, 'regressor__n_estimators': 300, 'regressor__max_depth': 6, 'regressor__learning_rate': 0.2, 'regressor__colsample_bytree': 1.0}
RMSE: 0.0618
R²: 0.9246

Model saved!


In [1]:
import pandas as pd
import joblib

model = joblib.load("best_xgboost_occupancy_rate_model_tuned.joblib")

test_data = pd.DataFrame({
    "POPULATION": [7790, 34635, 18500],
    "AREA": [1.4, 3.7, 2.8],
    "CAPACITY": [100, 250, 75],
    "Min_Temp_C": [-15.0, 5.0, -5.0],
    "Max_Temp_C": [-8.0, 12.0, 2.0],
    "Unemployment_Rate": [12.0, 8.5, 14.0],
})

test_data["Temp_Range"] = test_data["Max_Temp_C"] - test_data["Min_Temp_C"]

predictions = model.predict(test_data)

test_data["Predicted_OCCUPANCY_Rate"] = predictions
print(test_data[["POPULATION", "AREA","CAPACITY", "Min_Temp_C", "Unemployment_Rate", "Predicted_OCCUPANCY_Rate"]])

   POPULATION  AREA  CAPACITY  Min_Temp_C  Unemployment_Rate  \
0        7790   1.4       100       -15.0               12.0   
1       34635   3.7       250         5.0                8.5   
2       18500   2.8        75        -5.0               14.0   

   Predicted_OCCUPANCY_Rate  
0                  0.804162  
1                  1.041267  
2                  0.961446  


In [3]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Load the saved model
model = joblib.load("best_xgboost_occupancy_rate_model_tuned.joblib")

# Recreate the same data and split as training
model_df = pd.read_csv("TorontoShelterUsage2020_Final.csv")
model_df.replace([np.inf, -np.inf], np.nan, inplace=True)
model_df.dropna(subset=["OCCUPANCY_Rate"], inplace=True)
model_df["Temp_Range"] = model_df["Max_Temp_C"] - model_df["Min_Temp_C"]

X = model_df.drop(columns=["OCCUPANCY_Rate"])
y = model_df["OCCUPANCY_Rate"]

_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Predict and print metrics
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("\nModel Testing Results\n----------------------")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")


Model Testing Results
----------------------
RMSE: 0.0618
R²:   0.9246
